# 🔬 STCL — Single RP (Cav + Mon)
**Scanning Transfer Cavity Lock** · RedPitaya STEMlab 125-14 · `scan_mon` mode

One RedPitaya handles cavity scanning, monitoring, and (optionally) cavity locking.

| Board | Mode | Role |
|-------|------|------|
| `Cav` | `scan_mon` | Scans cavity · monitors signal · locks cavity length |
| `Lock1` *(optional)* | `lock` | Locks laser frequencies |

---
> **Wiring:** `Cav OUT2` → piezo amp → cavity · `Cav OUT1` → `Lock1 IN2` · photodiode → `Cav IN1`

---
## ⚙️ Phase 0: Configuration
<blockquote style="border-left:4px solid #e67e22; padding:6px 12px; background:#fdf6ec; color:#7f4f00; border-radius:4px;">
<strong>Edit only this section.</strong> All parameters flow through automatically.
</blockquote>

In [1]:
# ── Board IPs ─────────────────────────────────────────────────────────────────
RP_CAV_IP   = "192.168.0.99"    # Cav  — scan_mon RP  (required)
# RP_LOCK1_IP = "192.168.0.102" # Lock1 — laser lock RP (uncomment if present)
SSH_USER = "root"
SSH_PASS = "root"

# ── Scan parameters ────────────────────────────────────────────────────────────
CAV_DEC    = 32      # decimation: 8→1.0ms | 16→2.1ms | 32→4.2ms | 64→8.4ms
CAV_AMP    = 0.7     # V — triangle half-swing  (CAV_AMP + |CAV_OFFSET| ≤ 1.0 V)
CAV_OFFSET = 0.0     # V — DC offset

# ── Cavity (Master) lock parameters ───────────────────────────────────────────
CAV_RANGE     = [[0.15, 0.50], [1.70, 2.00]]  # ms — two reference peak windows
CAV_LOCKPOINT = 1.80                            # ms — target peak position
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Monitor ────────────────────────────────────────────────────────────────────
SHOW_TRIGGER = True   # True = dual-axis (cavity + trigger) | False = cavity only

# ── Slave 1 — Lock1 OUT1 ──────────────────────────────────────────────────────
SL1_LABEL     = "Laser_A"
SL1_RANGE     = [0.85, 1.10]   # ms
SL1_LOCKPOINT = 0.96            # ms
SL1_ENABLED   = True
SL1_PID       = {"P": 0.0, "I": 0.5, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Slave 2 — Lock1 OUT2 ──────────────────────────────────────────────────────
SL2_LABEL     = "Laser_B"
SL2_RANGE     = [0.50, 0.85]   # ms
SL2_LOCKPOINT = 0.60            # ms
SL2_ENABLED   = False           # set True when second laser is coupled in
SL2_PID       = {"P": 0.0, "I": 0.5, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Derived ────────────────────────────────────────────────────────────────────
_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3
print(f"Scan period : {_CAV_PERIOD_MS:.3f} ms  (dec={CAV_DEC})")
print(f"Amp / Offset: {CAV_AMP} V / {CAV_OFFSET} V")
print(f"Cav range   : {CAV_RANGE}  lockpoint: {CAV_LOCKPOINT} ms")
print(f"Trigger     : {'ON — dual-axis' if SHOW_TRIGGER else 'OFF — cavity only'}")

Scan period : 4.194 ms  (dec=32)
Amp / Offset: 0.7 V / 0.0 V
Cav range   : [[0.15, 0.5], [1.7, 2.0]]  lockpoint: 1.8 ms
Trigger     : ON — dual-axis


---
## 🔌 Phase 1: Upload & Connect
Imports, uploads RP-side scripts, connects SSH, starts PC event loop.

In [2]:
import sys, pathlib, threading, time

# Locate repo root
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

from lockclient import LockClient, RP_client, Monitor

# Apply trigger flag before any Monitor is constructed
Monitor.show_trigger = SHOW_TRIGGER

# Build RP dictionary
RPs = {
    "Cav": RP_client((RP_CAV_IP, 5000), {}, mode="scan_mon"),
    # "Lock1": RP_client((RP_LOCK1_IP, 5000), {}, mode="lock"),
}

print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Done.")
for name, rp in Lock.RPs.items():
    print(f"  {name:6s}  mode={rp.mode:10s}  addr={rp.addr[0]}")

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL
Uploading scripts and loading settings...
Done.
  Cav     mode=scan_mon    addr=192.168.0.99


In [3]:
# Connect boards (SSH → starts RunLock.py)
def _wrap(fn, err):
    try: fn()
    except Exception as exc: err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start(); t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(f"{name} timed out after {timeout_s}s — check board SSH")
    if "exc" in err:
        raise RuntimeError(f"{name} failed: {err['exc']}")

run_with_timeout(Lock.connect_all, timeout_s=45, name="connect_all")
print("All boards connected.")

connecting...
All boards connected.


In [4]:
# Start PC-side event loop + apply decimation
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running.")

Lock.set_dec("Cav", CAV_DEC)
print(f"Decimation: dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
for name, rp in Lock.RPs.items():
    print(f"  {name:6s}  {rp.addr[0]}  {'connected' if rp.connected else 'DISCONNECTED'}")

Event loop started.
Decimation: dec=32  period=4.194 ms
  Cav     192.168.0.99  connected


---
## 🔍 Phase 2: Signal Verification *(optional)*
Quick single-acquisition to verify wiring before scanning.

In [5]:
import numpy as np, matplotlib.pyplot as plt

acq = np.array(Lock.send("Cav", "acquire"))
if acq.size == 0:
    print("No data — check Cav is connected.")
else:
    t, ch1, ch2 = acq[0], acq[1], acq[2]
    fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
    axes[0].plot(t, ch1, lw=0.8, color="#4fc3f7"); axes[0].set_ylabel("IN1 [V]")
    axes[0].set_title("Cavity transmission"); axes[0].grid(True, alpha=0.3)
    axes[1].plot(t, ch2, lw=0.8, color="#a5d6a7"); axes[1].set_ylabel("IN2 [V]")
    axes[1].set_title("Trigger"); axes[1].set_xlabel("Time [ms]"); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"IN1: {ch1.min():.3f} … {ch1.max():.3f} V")
    print(f"IN2: {ch2.min():.3f} … {ch2.max():.3f} V")

IN1: 1.880 … 1.919 V
IN2: -0.063 … -0.005 V


---
## 📡 Phase 3: Cavity Scan + Live Monitor
Push settings → start scan → open monitor window.

> Stop the scan (`Lock.stop_loop("Cav")`) before starting the cavity lock (Phase 4).

In [6]:
# Push cavity settings
Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",   True)
Lock.update_setting("Cav", "Master", "PID",       CAV_PID)
print(f"Cavity settings pushed  range={CAV_RANGE}  lp={CAV_LOCKPOINT} ms")

check if lockpoint is still fine
Cavity settings pushed  range=[[0.15, 0.5], [1.7, 2.0]]  lp=1.8 ms


In [7]:
# Start cavity scan
Lock.start_scan("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)
print(f"Scan started — amp={CAV_AMP}V  offset={CAV_OFFSET}V  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f}ms")
time.sleep(3)   # wait for scan + port 5066 monitor server to start

Scan started — amp=0.7V  offset=0.0V  dec=32  period=4.194ms
connected to <socket.socket fd=2664, family=2, type=1, proto=0, laddr=('192.168.0.147', 61936), raddr=('192.168.0.99', 5065)>


In [8]:
# Start live monitor  (Qt window opens on main thread)
Lock.start_monitor("Cav")
print("Monitor started — Qt window should be visible.")
print("To stop: Lock.stop_monitor('Cav')")

Setting up monitor on main thread (Qt window)...
Settings added to plot
Monitor started (scan_mon mode — port 5066)
Monitor started — Qt window should be visible.
To stop: Lock.stop_monitor('Cav')


Settings added to plot
Settings added to plot
Settings added to plot


### 🔧 Tune scan / monitor settings while running
Edit values in the cell below and re-run to update without restarting.

In [9]:
# ── Edit and re-run to update live ────────────────────────────────────────────
CAV_DEC       = 32                            # change scan period
CAV_AMP       = 0.5                           # V — scan amplitude
CAV_OFFSET    = 0.0                           # V — scan offset
CAV_RANGE     = [[0.15, 0.50], [1.70, 2.00]] # ms — reference peak windows
CAV_LOCKPOINT = 1.80                          # ms — lockpoint

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

# Apply decimation (rescales time axis)
Lock.set_dec("Cav", CAV_DEC)

# Apply scan output (amplitude / offset)
if Lock.RPs["Cav"].loop_running:
    Lock.set_scan_output("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)

# Push cavity range + lockpoint (updates monitor markers automatically)
Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)

print(f"Updated — dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f}ms  amp={CAV_AMP}V  offset={CAV_OFFSET}V")
print(f"          range={CAV_RANGE}  lp={CAV_LOCKPOINT}ms")

check if lockpoint is still fine
Updated — dec=32  period=4.194ms  amp=0.5V  offset=0.0V
          range=[[0.15, 0.5], [1.7, 2.0]]  lp=1.8ms


In [10]:
# Stop / restart monitor
Lock.stop_monitor("Cav")
print("Monitor stopped.")
# Lock.start_monitor("Cav")  # uncomment to restart

Monitor stopped.


---
## 🔒 Phase 4: Cavity Lock
Stabilise cavity length via PID on the ramp offset.

> Stop the scan loop first — lock and scan cannot run simultaneously.

In [11]:
# Stop scan, then start cavity lock
if Lock.RPs["Cav"].loop_running:
    Lock.stop_loop("Cav")
    time.sleep(0.5)
    print("Scan stopped.")

# Refresh settings before locking
Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",   True)
Lock.update_setting("Cav", "Master", "PID",       CAV_PID)

Lock.start_lock("Cav")
print("Cavity lock started.  Stop with: Lock.stop_loop('Cav')")

Scan stopped.
check if lockpoint is still fine
Cavity lock started.  Stop with: Lock.stop_loop('Cav')


Exception occured during connection: [WinError 10057] A request to send or receive data was disallowed because the socket is not connected and (when sending on a datagram socket using a sendto call) no address was supplied


---
## 🔒 Phase 5: Laser Lock *(requires Lock1)*

In [ ]:
# Push Slave settings and start laser lock
if "Lock1" not in Lock.RPs:
    print("Lock1 not present — add it to RPs in Phase 1.")
else:
    Lock.update_setting("Lock1", "Slave1", "label",     SL1_LABEL)
    Lock.update_setting("Lock1", "Slave1", "range",     SL1_RANGE)
    Lock.update_setting("Lock1", "Slave1", "lockpoint", SL1_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave1", "enabled",   SL1_ENABLED)
    Lock.update_setting("Lock1", "Slave1", "PID",       SL1_PID)

    Lock.update_setting("Lock1", "Slave2", "label",     SL2_LABEL)
    Lock.update_setting("Lock1", "Slave2", "range",     SL2_RANGE)
    Lock.update_setting("Lock1", "Slave2", "lockpoint", SL2_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave2", "enabled",   SL2_ENABLED)
    Lock.update_setting("Lock1", "Slave2", "PID",       SL2_PID)

    Lock.start_lock("Lock1")
    print(f"Laser lock started.  Slave1={SL1_LABEL}  Slave2={SL2_LABEL}")
    print("Stop with: Lock.stop_loop('Lock1')")

---
## 📊 Phase 6: Error Monitor
Live frequency error (MHz) for each locked laser channel.

In [ ]:
# Switch from cavity monitor to error monitor
Lock.stop_monitor("Cav")
time.sleep(0.5)
Lock.start_error_monitor("Cav", tmin=20e-3)
print("Error monitor started.  Stop with: Lock.stop_monitor('Cav')")

In [ ]:
# Save recorded errors to JSON
# import time as _t
# filename = "lock_errors_{}".format(int(_t.time()))
# Lock.monitors["Cav"]["queue_err"].put(("save", filename))
# print("Saved to", filename + ".json")

---
## 🛑 Phase 7: Safe Shutdown
Always shut down in order: monitors → laser lock → cavity → disconnect.

> `Lock.close()` handles everything automatically in the correct order.

In [ ]:
# Full clean shutdown (recommended)
Lock.close()
print("All stopped and disconnected.")

In [ ]:
# ── Manual step-by-step shutdown (if needed) ──────────────────────────────────
# Lock.stop_monitor("Cav")
# time.sleep(0.5)
# if "Lock1" in Lock.RPs and Lock.RPs["Lock1"].loop_running:
#     Lock.stop_loop("Lock1"); time.sleep(0.5)
# if Lock.RPs["Cav"].loop_running:
#     Lock.stop_loop("Cav");  time.sleep(0.5)
# Lock.close()